# Python drills 7: errors, logging and testing

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-drills-07-errors-logging-and-testing.ipynb)

Companion to the [Errors, logging and testing](../learning/python-07-errors-logging-and-testing.ipynb) chapter. The exercises cover specific exceptions, retries, exception chaining, logging output and simple checks on your own functions.

Try each exercise in the code cell below its question. Use the sample inputs to compare your result with the expected output. Expand **Show solution** when you want to review the approach.

## Exercise 1: parse a percentage and raise a specific error

**Difficulty:** Medium · about 10 min

`InvalidPercentage` is given below. Write `parse_percentage(text)`. A text such as `"45%"` returns `0.45`. Text that is not a number followed by `%` raises `InvalidPercentage`.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `parse_percentage('45%')` | `0.45` |
| `parse_percentage('100%')` | `1.0` |
| `parse_percentage('x%')` | Raises `InvalidPercentage` |


*Hint: `float(text[:-1])` raises `ValueError` for `"x%"`. Catch that and raise your own error from it.*

In [ ]:
class InvalidPercentage(ValueError):
    """Raised for text that is not a percentage."""

def parse_percentage(text):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
class InvalidPercentage(ValueError):
    """Raised for text that is not a percentage."""

def parse_percentage(text):
    if not text.endswith("%"):
        raise InvalidPercentage(f"missing % sign: {text!r}")
    try:
        return float(text[:-1]) / 100
    except ValueError:
        raise InvalidPercentage(f"not a number: {text!r}") from None
```

</details>

## Exercise 2: retry until the call succeeds

**Difficulty:** Medium · about 10 min

Write `with_retries(fn, attempts)`. Call `fn()` up to `attempts` times. Return the first result that does not raise. If every attempt raises, raise the **last** error. `attempts` must be positive; otherwise raise `ValueError`.

**Sample input and expected output**

```python
calls = 0
def fetch():
    global calls
    calls += 1
    if calls < 3:
        raise ConnectionError("temporary")
    return "ok"

with_retries(fetch, 3)  # Expected: "ok", after three calls
with_retries(lambda: 0, 1)  # Expected: 0; a false-like result is still success
```

In [ ]:
def with_retries(fn, attempts):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def with_retries(fn, attempts):
    if attempts <= 0:
        raise ValueError("attempts must be positive")
    last_error = None
    for _ in range(attempts):
        try:
            return fn()
        except Exception as error:
            last_error = error
    raise last_error
```

</details>

## Exercise 3: log a load summary

**Difficulty:** Medium · about 10 min

Write `log_load_summary(logger_name, loaded, rejected)`. Log one line at `INFO` level with the text `loaded <loaded> rows, rejected <rejected>`. Use `%d` placeholders, not an f-string.

**Sample input and expected output**

```python
# With an INFO-level logging handler attached:
log_load_summary("practice.load", 120, 3)
# Expected message: loaded 120 rows, rejected 3
log_load_summary("practice.load", 0, 0)
# Expected message: loaded 0 rows, rejected 0
```

In [ ]:
import logging

def log_load_summary(logger_name, loaded, rejected):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
import logging

def log_load_summary(logger_name, loaded, rejected):
    logging.getLogger(logger_name).info("loaded %d rows, rejected %d", loaded, rejected)
```

</details>

## Exercise 4: clamp a value and validate the bounds

**Difficulty:** Medium · about 10 min

Write `clamp(value, low, high)`. Return `value` limited to the range from `low` to `high`, inclusive. If `low` is greater than `high`, raise `ValueError`.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `clamp(15, 0, 10)` | `10` |
| `clamp(-3, 0, 10)` | `0` |
| `clamp(5, 0, 10)` | `5` |

In [ ]:
def clamp(value, low, high):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def clamp(value, low, high):
    if low > high:
        raise ValueError("low must not exceed high")
    return max(low, min(value, high))
```

</details>

## Exercise 5: median

**Difficulty:** Medium · about 10 min

Write `median(values)`. Return the middle value of the sorted list. For an even number of values, return the mean of the two middle values. An empty list raises `ValueError`.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `median([3, 1, 2])` | `2` |
| `median([4, 1, 3, 2])` | `2.5` |
| `median([])` | Raises `ValueError` |

In [ ]:
def median(values):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def median(values):
    if not values:
        raise ValueError("median of an empty list")
    ordered = sorted(values)
    middle = len(ordered) // 2
    if len(ordered) % 2:
        return ordered[middle]
    return (ordered[middle - 1] + ordered[middle]) / 2
```

</details>

## Exercise 6: chain the original error

**Difficulty:** Medium · about 10 min

`LoadError` is given below. Write `load_int(text)`. It returns `int(text)`. If the text is not an integer, raise `LoadError` and keep the original `ValueError` as its cause.

**Sample input and expected output**

```python
load_int("42")  # Expected: 42
load_int("-9")  # Expected: -9
load_int("x")   # Raises LoadError; error.__cause__ is a ValueError
```

In [ ]:
class LoadError(Exception):
    """A value could not be loaded."""

def load_int(text):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
class LoadError(Exception):
    """A value could not be loaded."""

def load_int(text):
    try:
        return int(text)
    except ValueError as error:
        raise LoadError(f"not an integer: {text!r}") from error
```

</details>